# Resumen

## Librerías

In [14]:
import pandas as pd

## Datos

In [15]:
df_si = pd.read_csv('sell_in_suiza_geo.csv')
df_so = pd.read_csv('limpio/sell_out_suiza_v2.csv')

C:\Users\manue\AppData\Local\Temp\ipykernel_30332\1014752903.py:2: DtypeWarning: Columns (0: FSDCUSTOMERNAME, 1: FSDCUSTOMERFRIENDLYNAME, 2: ROLE) have mixed types. Specify dtype option on import or set low_memory=False.
  df_so = pd.read_csv('limpio/sell_out_suiza_v2.csv')


### Visualización de datos iniciales

In [16]:
for nombre, df in [('SELL IN', df_si), ('SELL OUT', df_so)]:
    print(f'\n{"="*40}')
    print(f'{nombre}')
    print(f'{"="*40}')
    print(f'Shape: {df.shape}')
    print(f'\nNulos por columna (solo las que tienen):')
    nulos = df.isnull().sum()
    print(nulos[nulos > 0])
    print(f'\nTipos de dato:')
    print(df.dtypes)


SELL IN
Shape: (294132, 51)

Nulos por columna (solo las que tienen):
SHIP_TO_CUST_ID         1013
SHIP_TO_NAME            1013
SHIP_TO_NAME_V2         1013
FSD                   268006
MATL_LEVEL_5_CD         1793
MATL_LEVEL_5_DESC       1793
OE_TEXT               256870
UUHP_TIRE_CD          251285
UUHP_TIRE_DESC        251285
CUSTOMERID             36214
CUSTOMERPOSTALCODE      2802
CUSTOMERLATITUDE        2802
CUSTOMERLONGITUDE       2802
dtype: int64

Tipos de dato:
CLUSTER_VIEW_LEVEL_1             str
CLUSTER_VIEW_LEVEL_2             str
CLUSTER_VIEW_LEVEL_3             str
CLUSTER_VIEW_LEVEL_4             str
CUST_HIER_LVL_0_ID               str
CUST_HIER_LVL_0_DESC             str
CUST_HIER_LVL_1_ID               str
CUST_HIER_LVL_1_DESC             str
CUST_HIER_LVL_2_ID               str
CUST_HIER_LVL_2_DESC             str
CUST_HIER_LVL_3_ID               str
CUST_HIER_LVL_3_DESC             str
CUST_HIER_LVL_4_ID               str
CUST_HIER_LVL_4_DESC             str
CUST_

##### Correción de tipo de datos iniciales

In [17]:
# SELL IN - corregir tipos
df_si['MONTH_DT'] = pd.to_datetime(df_si['MONTH_DT'])
df_si['CUSTOMERLATITUDE'] = df_si['CUSTOMERLATITUDE'].str.replace(',', '.').astype(float)
df_si['CUSTOMERLONGITUDE'] = df_si['CUSTOMERLONGITUDE'].str.replace(',', '.').astype(float)
df_si['CUSTOMERPOSTALCODE'] = df_si['CUSTOMERPOSTALCODE'].dropna().astype(int)

# SELL OUT - corregir tipos
df_so['MONTHDATE'] = pd.to_datetime(df_so['MONTHDATE'])
df_so['POSTALCODE'] = df_so['POSTALCODE'].dropna().astype(int)

# Verificar rangos temporales
print('Sell In - rango fechas:')
print(df_si['MONTH_DT'].min(), '→', df_si['MONTH_DT'].max())

print('\nSell Out - rango fechas:')
print(df_so['MONTHDATE'].min(), '→', df_so['MONTHDATE'].max())

Sell In - rango fechas:
2023-01-01 00:00:00 → 2026-03-01 00:00:00

Sell Out - rango fechas:
2018-01-01 00:00:00 → 2026-02-01 00:00:00


## Distribución de registros por año

In [ ]:
print('Sell In - registros por año:')
print(df_si['MONTH_DT'].dt.year.value_counts().sort_index())

print('\nSell Out - registros por año:')
print(df_so['MONTHDATE'].dt.year.value_counts().sort_index())

Sell In - registros por año:
MONTH_DT
2023    87984
2024    95096
2025    92111
2026    18941
Name: count, dtype: int64

Sell Out - registros por año:
MONTHDATE
2018      4938
2019     18846
2020     45988
2021     71590
2022    107783
2023    130876
2024    147020
2025    152645
2026     25013
Name: count, dtype: int64


### Outliers o errores

In [20]:
print('=== SELL IN - BILLED_QTY ===')
print(df_si['BILLED_QTY'].describe())
print('Negativos:', (df_si['BILLED_QTY'] < 0).sum())

print('\n=== SELL OUT - TOTALSELLOUTQTTY ===')
print(df_so['TOTALSELLOUTQTTY'].describe())
print('Negativos:', (df_so['TOTALSELLOUTQTTY'] < 0).sum())

=== SELL IN - BILLED_QTY ===
count    294132.000000
mean          5.184852
std          25.639550
min        -287.000000
25%           0.000000
50%           4.000000
75%           4.000000
max        2917.000000
Name: BILLED_QTY, dtype: float64
Negativos: 6771

=== SELL OUT - TOTALSELLOUTQTTY ===
count    704699.000000
mean          2.423027
std          21.008554
min        -168.000000
25%           0.000000
50%           0.000000
75%           2.000000
max        3382.000000
Name: TOTALSELLOUTQTTY, dtype: float64
Negativos: 4409


##### Outliers altos en Sell-In y Sell-Out

In [21]:
# Sell In
print('=== Top 5 registros por BILLED_QTY (Sell In) ===')
cols_si = ['MONTH_DT', 'PAYER_CUSTOMER_NAME', 'CUSTOMERPOSTALCODE', 
           'MATL_LEVEL_5_DESC', 'BILLED_QTY']
print(df_si.nlargest(5, 'BILLED_QTY')[cols_si].to_string())

# Sell Out
print('\n=== Top 5 registros por TOTALSELLOUTQTTY (Sell Out) ===')
cols_so = ['MONTHDATE', 'CUSTOMERNAME', 'POSTALCODE', 
           'BRAND', 'TOTALSELLOUTQTTY']
print(df_so.nlargest(5, 'TOTALSELLOUTQTTY')[cols_so].to_string())

=== Top 5 registros por BILLED_QTY (Sell In) ===
         MONTH_DT PAYER_CUSTOMER_NAME  CUSTOMERPOSTALCODE MATL_LEVEL_5_DESC  BILLED_QTY
164599 2024-04-01         Empresa_2_C              4624.0         Passenger      2917.0
167817 2024-03-01         Empresa_2_C              4624.0         Passenger      2831.0
251381 2023-06-01         Entidad_674              4553.0         Passenger      2800.0
70359  2025-09-01          Entidad_24              8107.0         Passenger      1775.0
110347 2023-02-01         Empresa_2_C              4624.0         Passenger      1700.0

=== Top 5 registros por TOTALSELLOUTQTTY (Sell Out) ===
        MONTHDATE CUSTOMERNAME  POSTALCODE    BRAND  TOTALSELLOUTQTTY
580987 2024-08-01    Empresa_2      4624.0      NaN            3382.0
500547 2023-05-01    Empresa_2      4624.0  brand_4            3160.0
469164 2022-10-01    Empresa_2      4624.0      NaN            3021.0
423498 2020-05-01    Empresa_2      4624.0  brand_4            2780.0
626699 2025-03-0

### Valores negativos en Sell_In y Sell_Out

In [22]:
print('=== Negativos Sell In ===')
print('Por año:')
print(df_si[df_si['BILLED_QTY'] < 0]['MONTH_DT'].dt.year.value_counts().sort_index())
print('\nVolumen total negativo:', df_si[df_si['BILLED_QTY'] < 0]['BILLED_QTY'].sum())
print('% sobre total:', round(df_si[df_si['BILLED_QTY'] < 0]['BILLED_QTY'].sum()/df_si['BILLED_QTY'].sum()*100, 1))

print('\n=== Negativos Sell Out ===')
print('BRAND nulo en negativos:')
print(df_so[df_so['TOTALSELLOUTQTTY'] < 0]['BRAND'].isnull().sum(), 'de', (df_so['TOTALSELLOUTQTTY'] < 0).sum())
print('\nVolumen total negativo:', df_so[df_so['TOTALSELLOUTQTTY'] < 0]['TOTALSELLOUTQTTY'].sum())
print('% sobre total:', round(df_so[df_so['TOTALSELLOUTQTTY'] < 0]['TOTALSELLOUTQTTY'].sum()/df_so['TOTALSELLOUTQTTY'].sum()*100, 1))

=== Negativos Sell In ===
Por año:
MONTH_DT
2023    2003
2024    1694
2025    2376
2026     698
Name: count, dtype: int64

Volumen total negativo: -35704.0
% sobre total: -2.3

=== Negativos Sell Out ===
BRAND nulo en negativos:
1730 de 4409

Volumen total negativo: -16019.0
% sobre total: -0.9
